Search EAF annotation text by word or phrase. Matches are literal substrings, with Unicode accents normalized.

In [ ]:
from eaf_search import search_eaf

matches = search_eaf("data/eaf_example.eaf", "ĩyẽ")
matches

In [ ]:
# Search a phrase within the transcription tier only.
search_eaf("data/eaf_example.eaf", "ĩyẽ mʉ̃ãrẽ", tier="JBMassa-des")

Count whitespace-separated words in the transcription tier. Accents, punctuation, and case are preserved; equivalent Unicode spellings are combined. Total words counts occurrences; unique words counts distinct spellings.

In [ ]:
from eaf_search import count_eaf_words

word_counts = count_eaf_words("data/eaf_example.eaf", tier="JBMassa-des")
print("Total words:", sum(word_counts.values()))
print("Unique words:", len(word_counts))
word_counts.most_common()  # (word, count), most frequent first

In [ ]:
# For multiple EAF files with the same transcription tier:
# from pathlib import Path
# word_counts = count_eaf_words(sorted(Path("data").glob("*.eaf")), tier="JBMassa-des")

Display the transcription and its Portuguese translation as columns. Rows are joined by annotation reference; untranslated segments remain in the table with a blank translation.

In [ ]:
import importlib
import eaf_search
import pandas as pd

importlib.reload(eaf_search)  # Pick up changes if already imported.
rows = eaf_search.eaf_translation_rows("data/eaf_example.eaf")
df = pd.DataFrame(rows, columns=["annotation_id", "start_ms", "end_ms", "text", "translation"])
df.head(10)

Segment table: one row per transcription interval, with translation, morphemes, glosses, and metadata. associated_annotations retains child IDs, timing, parent links, and tier metadata. Unknown child times remain None.

In [ ]:
from eaf_dataframe import eaf_dataframe

df = eaf_dataframe("data/eaf_example.eaf")
display(df[["timestamp", "end_timestamp", "transcript", "translation", "morphemes", "glosses", "participant"]])

In [ ]:
# Inspect every annotation associated with the second segment.
df.loc[1, "associated_annotations"]

Attribute table: one row per annotation, original XML attribute names as columns, and start timestamp as the index. Repeated timestamps represent separate annotations. NaT indicates unknown timing. ANNOTATION_VALUE is the annotation text, included alongside the attributes.

In [ ]:
from eaf_dataframe import eaf_attribute_dataframe

attributes_df = eaf_attribute_dataframe("data/eaf_example.eaf")
display(attributes_df)

Search Portuguese translations or original-language transcripts. Whole-word matching is the default; use whole_word=False for substrings. Audio search returns annotated source-language segment intervals, with matching morpheme intervals where available. It does not recognize untranscribed speech.

In [ ]:
from eaf_dataframe import eaf_dataframe
from corpus_search import search_translation, search_transcript, search_audio

df = eaf_dataframe("data/eaf_example.eaf")
columns = ["timestamp", "end_timestamp", "transcript", "translation"]

# Portuguese: estoria also matches estória.
display(search_translation(df, "estoria")[columns])

In [ ]:
# Original-language word or phrase (accents preserved).
display(search_transcript(df, "ĩyẽ")[columns])

In [ ]:
# Audio intervals for an original-language word.
audio_matches = search_audio(df, "ĩyẽ")
display(audio_matches[columns + ["start_seconds", "end_seconds", "timing_scope", "matching_child_intervals"]])

# Locate source audio segments using their Portuguese translation.
display(search_audio(df, "estoria", language="translation")[columns + ["start_seconds", "end_seconds"]])

Persistent lexicon / word bank. Running add saves to data/lexicon.json. Examples include source file, transcript, sentence translation, segment timestamps and available child intervals. Enter lexical meanings yourself; sentence translations are not word definitions. Re-adding a word merges examples without duplicates.

In [ ]:
from lexicon import Lexicon
from eaf_dataframe import eaf_dataframe

df = eaf_dataframe("data/eaf_example.eaf")
lexicon = Lexicon("data/lexicon.json")

# Save a word with its automatically identified corpus examples.
entry = lexicon.add(
    "ĩyẽ", df,
    source_file="data/eaf_example.eaf",
    notes="Meaning to be reviewed",
)
display(lexicon.to_dataframe())
display(lexicon.examples("ĩyẽ"))

In [ ]:
# Update a meaning when you have established it:
# lexicon.add("ĩyẽ", df, source_file="data/eaf_example.eaf",
#             meaning="Enter the verified meaning here", part_of_speech="...")

# Optionally add only selected matching examples:
# lexicon.add("ĩyẽ", df, source_file="data/eaf_example.eaf", example_ids=["a1"])

# Reopen the same file in a later notebook session:
# lexicon = Lexicon("data/lexicon.json")
# display(lexicon.to_dataframe())